In [3]:
import requests
from bs4 import BeautifulSoup

url = "https://freemusicarchive.org/genre/Jazz?pageSize=50&page=1&search-genre=Jazz&sort=_score&d=0"
headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(url, headers=headers)
soup = BeautifulSoup(r.text, "html.parser")

# Show first 10 links with attributes
download_links = soup.find_all("a", {"title": "Download"})

for a in soup.find_all("a", {"title": "Download"})[:10]:
    print(a.get("data-url"), "\n---")

https://freemusicarchive.org/track/01_What_A_Day/downloadOverlay/ 
---
https://freemusicarchive.org/track/06_Shasha/downloadOverlay/ 
---
https://freemusicarchive.org/track/did-you-see-my-budgie-tp-062/downloadOverlay/ 
---
https://freemusicarchive.org/track/Santa_on_a_Segway/downloadOverlay/ 
---
https://freemusicarchive.org/track/Arana/downloadOverlay/ 
---
https://freemusicarchive.org/track/Blue_bossa/downloadOverlay/ 
---
https://freemusicarchive.org/track/Song_for_Bilbao/downloadOverlay/ 
---
https://freemusicarchive.org/track/Kurina_blues/downloadOverlay/ 
---
https://freemusicarchive.org/track/C-mol_blues/downloadOverlay/ 
---
https://freemusicarchive.org/track/Balkan_improvisation/downloadOverlay/ 
---


In [8]:
links = soup.find_all("a", {"title": "Download"})
genre = "Jazz"

for idx, a in enumerate(links):
    download_url = a.get('data-url').replace("downloadOverlay", "download")
    track_slug = download_url.split("/track/")[1].split("/download")[0]
    filename = f"track-{genre}-{idx+1}.mp3"
    print(track_slug, filename)

01_What_A_Day track-Jazz-1.mp3
06_Shasha track-Jazz-2.mp3
did-you-see-my-budgie-tp-062 track-Jazz-3.mp3
Santa_on_a_Segway track-Jazz-4.mp3
Arana track-Jazz-5.mp3
Blue_bossa track-Jazz-6.mp3
Song_for_Bilbao track-Jazz-7.mp3
Kurina_blues track-Jazz-8.mp3
C-mol_blues track-Jazz-9.mp3
Balkan_improvisation track-Jazz-10.mp3
Caravan track-Jazz-11.mp3
Konflikt track-Jazz-12.mp3
Blue_Monk track-Jazz-13.mp3
WM030-07-SidPeacock07mp3 track-Jazz-14.mp3
02_East_Of_Jaffa_-_The_Father_The track-Jazz-15.mp3
The_Great_Red_Spot track-Jazz-16.mp3
WM030-03-SidPeacock03mp3 track-Jazz-17.mp3
WM030-04-SidPeacock04mp3 track-Jazz-18.mp3
WM030-02-SidPeacock02mp3 track-Jazz-19.mp3
WM030-05-SidPeacock05mp3 track-Jazz-20.mp3
WM030-06-SidPeacock06mp3 track-Jazz-21.mp3
WM030-01-SidPeacock01mp3 track-Jazz-22.mp3
Pt_I track-Jazz-23.mp3
Avenue_B track-Jazz-24.mp3
No_More_Blues track-Jazz-25.mp3
hearing-voices track-Jazz-26.mp3
saturns-rings track-Jazz-27.mp3
andromeda-1 track-Jazz-28.mp3
plutos-moons track-Jazz-29.mp3


In [ ]:
import os
import shutil
import requests
from bs4 import BeautifulSoup
import pandas as pd
from tqdm import tqdm

# Your genre → URL mapping
genre_urls = {
    "blues": "https://freemusicarchive.org/genre/Blues?pageSize=50&page=1&search-genre=Blues&sort=_score&d=0",
    "jazz": "https://freemusicarchive.org/genre/Jazz?pageSize=50&page=1&search-genre=Jazz&sort=_score&d=0",
    "country": "https://freemusicarchive.org/genre/Country?pageSize=50&page=1&search-genre=Country&sort=_score&d=0",
    "pop": "https://freemusicarchive.org/genre/Pop?pageSize=50&page=1&search-genre=Pop&sort=_score&d=0",
    "lofi": "https://freemusicarchive.org/genre/Lo-fi-Instrumental?pageSize=20&page=1&search-genre=Lo-fi%20Instrumental&sort=_score&d=0",
    "rock-garage": "https://freemusicarchive.org/genre/Garage?pageSize=50&page=1&search-genre=Garage&sort=_score&d=0",
    "rock-goth": "https://freemusicarchive.org/genre/Goth?pageSize=50&page=1&search-genre=Goth&sort=_score&d=0",
    "rock-industrial": "https://freemusicarchive.org/genre/Industrial?pageSize=50&page=1&search-genre=Industrial&sort=_score&d=0",
    "rock-krautrock": "https://freemusicarchive.org/genre/Krautrock?pageSize=50&page=1&search-genre=Krautrock&sort=_score&d=0",
    "rock-punk": "https://freemusicarchive.org/genre/Punk?pageSize=50&page=1&search-genre=Punk&sort=_score&d=0",
    "metal": "https://freemusicarchive.org/genre/Metal?pageSize=50&page=1&search-genre=Metal&sort=_score&d=0",
    "rnb": "https://freemusicarchive.org/genre/Soul-RB?pageSize=50&page=1&search-genre=Soul-RnB&sort=_score&d=0",
    "folk": "https://freemusicarchive.org/genre/Folk?pageSize=50&page=1&search-genre=Folk&sort=_score&d=0",
    "classical": "https://freemusicarchive.org/genre/Classical?pageSize=50&page=1&search-genre=Classical&sort=_score&d=0",
    "hiphop": "https://freemusicarchive.org/genre/Hip-Hop?pageSize=50&page=1&search-genre=Hip-Hop&sort=_score&d=0",
}

base_dir = os.path.expanduser("~/data/project/music")
os.makedirs(base_dir, exist_ok=True)

metadata = []

headers = {"User-Agent": "Mozilla/5.0"}

for genre, url in genre_urls.items():
    print(f"\nProcessing genre: {genre}")

    genre_dir = os.path.join(base_dir, genre)
    os.makedirs(genre_dir, exist_ok=True)

    # Remove old mp3 files
    for f in os.listdir(genre_dir):
        if f.endswith(".mp3"):
            os.remove(os.path.join(genre_dir, f))

    # Fetch genre page
    r = requests.get(url, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    # Extract all download links
    download_links = soup.find_all("a", {"title": "Download"})

    for idx, link in enumerate(tqdm(download_links, desc=genre)):
        data_url = link.get("data-url")
        if not data_url:
            continue
        # Fix URL
        download_url = data_url.replace("downloadOverlay", "download")

        # Track name (closest sibling span with class track-name)
        track_tag = link.find_parent("li")
        track_name = "unknown"
        if track_tag:
            name_tag = track_tag.find("span", class_="ptxt-track")
            if name_tag:
                track_name = name_tag.get_text(strip=True)

        # Save file
        filename = f"track-{genre}-{idx+1}.mp3"
        filepath = os.path.join(genre_dir, filename)

        try:
            resp = requests.get(download_url, headers=headers, stream=True)
            with open(filepath, "wb") as f:
                shutil.copyfileobj(resp.raw, f)
            metadata.append({"file_name": filename, "track_name": track_name})
        except Exception as e:
            print(f"Failed to download {download_url}: {e}")

# Save metadata
csv_path = os.path.join(base_dir, "metadata.csv")
pd.DataFrame(metadata).to_csv(csv_path, index=False)
print(f"\nMetadata saved at {csv_path}")
